In [70]:
import os
import time
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import errors
from google.genai import types
from pypdf import PdfReader

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

GENERATION_MODELS = ["gemini-flash-lite-latest", "gemini-flash-latest", "gemini-3.8-flash"]
EMBEDDING_MODEL = "gemini-embedding-001"


def with_retry(operation, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return operation()
        except errors.APIError as e:
            if e.code == 429:
                print("Quota exceeded, waiting 60 seconds...")
                time.sleep(60)
            elif e.code >= 500:
                wait = 2 ** attempt
                print(f"Server busy, waiting {wait} seconds...")
                time.sleep(wait)
            else:
                raise
    raise Exception("Operation failed, try again in a few minutes.")


def embed(texts):
    result = with_retry(lambda: client.models.embed_content(
        model=EMBEDDING_MODEL, contents=texts))
    return [e.values for e in result.embeddings]


def generate(prompt, schema=None, rounds=3):
    config = types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json" if schema else None,
        response_schema=schema,
    )
    for r in range(rounds):
        for model in GENERATION_MODELS:
            try:
                response = client.models.generate_content(model=model, contents=prompt, config=config)
                if schema is None:
                    return response.text
                return response.parsed or schema.model_validate_json(response.text)
            except errors.APIError as e:
                print(f"{model} unavailable ({e.code})")
        wait = 5 * (r + 1)
        print(f"All models busy, retrying in {wait} seconds...")
        time.sleep(wait)
    raise Exception("No model responded, try again in a few minutes.")


def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


print("Ready!")

Ready!


In [71]:
import re

PDF_PATH = "documents/hr_regulation.pdf"
SECTION_START = re.compile(r"^(\d+(?:\.\d+)*)\.\s+(.*)")
MIN_SECTION_LENGTH = 30


def read_sections(pdf_path):
    reader = PdfReader(pdf_path)
    sections = []
    current = None
    for page_no, page in enumerate(reader.pages, start=1):
        for line in (page.extract_text() or "").split("\n"):
            line = line.strip()
            if not line or re.fullmatch(r"Sayfa \d+", line):
                continue
            match = SECTION_START.match(line)
            if match:
                current = {"number": match.group(1), "text": line, "page": page_no}
                sections.append(current)
            elif current:
                current["text"] += " " + line
    return sections


def short_title(text, max_length=80):
    title = re.split(r"[:;]", text, maxsplit=1)[0]
    return title[:max_length].strip()


def build_chunks(sections):
    titles = {s["number"]: short_title(s["text"]) for s in sections}
    chunks = []
    for index, s in enumerate(sections):
        if len(s["text"]) < MIN_SECTION_LENGTH:
            continue  # skip heading-only sections, their content lives in children
        parts = s["number"].split(".")
        parent_numbers = [".".join(parts[:i]) for i in range(1, len(parts))]
        breadcrumb = " > ".join(titles[n] for n in parent_numbers if n in titles)
        chunks.append({
            "id": f"{index:04d}-{s['number']}",
            "text": f"{breadcrumb}\n{s['text']}" if breadcrumb else s["text"],
            "section": s["number"],
            "page": s["page"],
        })
    return chunks


sections = read_sections(PDF_PATH)
chunks = build_chunks(sections)
print(f"{len(sections)} sections found, {len(chunks)} chunks created.")
print("\nExample chunk:")
print(next(c["text"] for c in chunks if c["section"] == "5.7.5.1.1"))

266 sections found, 238 chunks created.

Example chunk:
5. İSTİHDAM ŞEKİLLERİ, POZİSYONLARIN TESPİTİ VE GRUPLANDIRILMASI > 5.7. Çalışma Saatleri, Fazla Çalışma ve İzinler > 5.7.5. Mazeret izni > 5.7.5.1. Doğum İzni
5.7.5.1.1. Evlenmesi veya eşinin doğum yapması halinde üç gün,


In [72]:
import chromadb

DB_PATH = "chroma_db"
COLLECTION_NAME = "hr_regulation"

db = chromadb.PersistentClient(path=DB_PATH)
collection = db.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

if collection.count() > 0:
    print(f"Collection already has {collection.count()} chunks, skipping indexing.")
else:
    BATCH_SIZE = 20
    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]
        collection.add(
            ids=[c["id"] for c in batch],
            documents=[c["text"] for c in batch],
            embeddings=embed([c["text"] for c in batch]),
            metadatas=[{"section": c["section"], "page": c["page"]} for c in batch],
        )
        print(f"{min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks indexed")
        if i + BATCH_SIZE < len(chunks):
            time.sleep(15)  # stay under the free tier limit
    print("Indexing complete.")

Collection already has 238 chunks, skipping indexing.


In [73]:
from typing import Literal
from pydantic import BaseModel, Field

NO_ANSWER_TEXT = "Bu konuda bilgim yok. Detaylı bilgi için İnsan Kaynakları birimine başvurabilirsiniz."
MISSING_INFO_TEXT = ("Yönetmelikte şu konularda bilgi bulunmuyor: {items}. "
                     "Detaylı bilgi için İnsan Kaynakları birimine başvurabilirsiniz.")
HELP_TEXT = ("Merhaba! Şirketin İnsan Kaynakları Yönetmeliği hakkındaki sorularınızı yanıtlayabilirim. "
             "Örneğin yıllık izin, mazeret ve doğum izinleri, çalışma saatleri, deneme süresi veya "
             "disiplin süreçleri hakkında sorabilirsiniz.")
THANKS_TEXT = "Rica ederim! Başka bir sorunuz olursa yardımcı olmaktan memnuniyet duyarım."


class MessageIntent(BaseModel):
    intent: Literal["hr_question", "greeting_or_help", "thanks_or_goodbye", "other"]


ROUTER_INSTRUCTIONS = """Classify the user's message sent to a company HR assistant.
- hr_question: asks about work rules, leave, working hours, pay, benefits, discipline or other employment matters.
- greeting_or_help: greetings, or asking what the assistant can do.
- thanks_or_goodbye: thanking the assistant or ending the conversation.
- other: anything else."""

REPLIES = {"greeting_or_help": HELP_TEXT, "thanks_or_goodbye": THANKS_TEXT}


def refusal(question):
    # Called only when we are about to refuse, so normal HR questions cost no extra request
    routed = generate(f"{ROUTER_INSTRUCTIONS}\n\nMessage: {question}", schema=MessageIntent)
    text = REPLIES.get(routed.intent, NO_ANSWER_TEXT)
    return {"answer": text, "sections": [], "refused": True, "missing_info": []}


class HRAnswer(BaseModel):
    can_answer: bool = Field(description="False only if no part of the question can be answered from the excerpts.")
    answer: str = Field(description="Answer in Turkish, addressed to the employee. No section numbers.")
    used_sections: list[str] = Field(description="Section numbers of the excerpts actually used in the answer.")
    missing_info: list[str] = Field(description="Parts of the question the excerpts do not cover, as short Turkish phrases.")


INSTRUCTIONS = """You are an HR assistant for the employees of a company.
You answer questions using only the regulation excerpts given below.

How to read the excerpts:
- Each excerpt has a section number, a section path and the section text.
- The section path only shows where the section sits in the document. Section titles can be
  imprecise, so take the rule, who it applies to and its conditions from the section text itself.
- Some excerpts may be unrelated to the question. Use only the ones that answer it and
  ignore the others without mentioning them.

How to answer:
- Find every separate thing the person is asking and answer each of them.
- Consider everyone the question is about: the employee and anyone they mention.
  Include every right in the excerpts that applies to their situation.
- Do not add rules for situations or people the question does not mention.
- Never guess, never use outside knowledge (for example general labour law) and never
  calculate amounts the excerpts do not state. Put such parts in missing_info instead.
- Set can_answer to false only if no part of the question can be answered from the excerpts.
- Write the answer in clear, plain Turkish. Do not put section numbers in the answer text."""


def search(question, top_k=5, threshold=0.6):
    result = collection.query(query_embeddings=embed(question), n_results=top_k)
    hits = []
    for text, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        similarity = 1 - distance  # Chroma returns cosine distance
        if similarity >= threshold:
            hits.append({"text": text, "section": meta["section"], "page": meta["page"], "score": similarity})
    return hits


def split_chunk(text):
    # Stored chunks are "breadcrumb\nsection text"; top-level sections have no breadcrumb
    path, _, body = text.rpartition("\n")
    return path, body


def format_context(hits):
    blocks = []
    for h in hits:
        path, body = split_chunk(h["text"])
        blocks.append(f"[Section {h['section']}]\nPath: {path or '-'}\nText: {body}")
    return "\n\n".join(blocks)


def answer_question(question, top_k=5, threshold=0.6):
    hits = search(question, top_k, threshold)
    if not hits:
        return refusal(question)

    prompt = f"{INSTRUCTIONS}\n\nExcerpts:\n{format_context(hits)}\n\nQuestion: {question}"
    result = generate(prompt, schema=HRAnswer)

    if not result.can_answer:
        return refusal(question)

    answer = result.answer.strip()
    if result.missing_info:
        answer += "\n\n" + MISSING_INFO_TEXT.format(items=", ".join(result.missing_info))

    retrieved = {h["section"] for h in hits}
    sections = [s for s in result.used_sections if s in retrieved]  # drop invented section numbers
    return {"answer": answer, "sections": sections, "refused": False, "missing_info": result.missing_info}


def ask(question, top_k=5, threshold=0.6):
    result = answer_question(question, top_k, threshold)
    print("Question:", question)
    print("Sources:", result["sections"] or "-")
    print("\nAnswer:", result["answer"])
    print("-" * 50)


ask("Evlenirsem kaç gün izin alırım?")

Question: Evlenirsem kaç gün izin alırım?
Sources: ['5.7.5.1.1']

Answer: Evlenmeniz halinde üç gün mazeret izni alabilirsiniz.
--------------------------------------------------


In [78]:

ask("merhaba nasılsınız bana hangi konularda yardımcı olabilirsiniz?")
ask("teşekkürler, sorularıma almanca cevap verir misin?")
ask("hello my wife is also working here, how much leave will we get and can you calculate my wife's severance after pregnancy?")

Question: merhaba nasılsınız bana hangi konularda yardımcı olabilirsiniz?
Sources: -

Answer: Merhaba! Şirketin İnsan Kaynakları Yönetmeliği hakkındaki sorularınızı yanıtlayabilirim. Örneğin yıllık izin, mazeret ve doğum izinleri, çalışma saatleri, deneme süresi veya disiplin süreçleri hakkında sorabilirsiniz.
--------------------------------------------------
Question: teşekkürler, sorularıma almanca cevap verir misin?
Sources: -

Answer: Bu konuda bilgim yok. Detaylı bilgi için İnsan Kaynakları birimine başvurabilirsiniz.
--------------------------------------------------
Question: hello my wife is also working here, how much leave will we get and can you calculate my wife's severance after pregnancy?
Sources: ['5.7.5.1', '5.7.5.1.1', '5.7.8.3']

Answer: Eşinizin doğum yapması halinde üç gün mazeret izni alabilirsiniz. Doğum yapan personele, isteği halinde doğum öncesi sekiz hafta ve doğum sonrası sekiz hafta olmak üzere doğum izni verilir ve bu süreden sonra çocuğu bir yaşına gelinc

In [75]:
import json
import re

SECTION_NUMBER = re.compile(r"\d+(?:\.\d+)+")


def normalize(text):
    # Turkish-aware lowercase: "İ" -> "i", "I" -> "ı"
    return text.replace("İ", "i").replace("I", "ı").lower()


def contains_keyword(answer, keyword):
    # Remove section numbers like 5.7.5.1.3 so their digits don't count as matches
    text = SECTION_NUMBER.sub(" ", answer)
    # Must start at a word boundary; may be followed by Turkish suffixes ("2 aydır"),
    # but not by more digits ("14" must not match "140" or "14,5")
    pattern = rf"(?<![\w,]){re.escape(normalize(keyword))}(?!,?\d)"
    return re.search(pattern, text) is not None


def evaluate_case(case):
    result = answer_question(case["question"])
    answer = normalize(result["answer"])

    if case.get("expect_no_answer"):
        passed = result["refused"] or bool(result["missing_info"])
        detail = "refused" if result["refused"] else "flagged as not covered"
        return {"id": case["id"], "passed": passed, "retrieval": None,
                "detail": detail if passed else "answered as if the document covered it"}

    expected = case["expected_sections"]
    retrieval = len([s for s in expected if s in result["sections"]]) / len(expected)

    if result["refused"]:
        return {"id": case["id"], "passed": False, "retrieval": retrieval, "detail": "refused but answer exists"}

    missing = [group for group in case["expected_keywords"]
               if not any(contains_keyword(answer, word) for word in group)]
    unwanted = [word for word in case.get("forbidden_keywords", []) if normalize(word) in answer]

    problems = []
    if missing:
        problems.append(f"missing: {missing}")
    if unwanted:
        problems.append(f"unwanted: {unwanted}")

    passed = not problems
    return {"id": case["id"], "passed": passed, "retrieval": retrieval,
            "detail": "ok" if passed else "; ".join(problems)}


def run_evaluation(path="eval/test_cases.json", pause_seconds=4):
    with open(path, encoding="utf-8") as f:
        cases = json.load(f)

    results = []
    for case in cases:
        r = evaluate_case(case)
        results.append(r)
        status = "PASS" if r["passed"] else "FAIL"
        retrieval = "-" if r["retrieval"] is None else f"{r['retrieval']:.0%}"
        print(f"[{status}] {r['id']:<32} retrieval: {retrieval:<5} {r['detail']}")
        time.sleep(pause_seconds)  # stay under the free tier request limit

    passed = sum(r["passed"] for r in results)
    scored = [r["retrieval"] for r in results if r["retrieval"] is not None]
    print("-" * 72)
    print(f"Answers correct: {passed}/{len(results)}")
    print(f"Average retrieval recall: {sum(scored) / len(scored):.0%}")
    return results


results = run_evaluation()

[PASS] annual-leave-5-years             retrieval: 100%  ok
[PASS] marriage-leave                   retrieval: 100%  ok
[PASS] probation-period                 retrieval: 100%  ok


KeyboardInterrupt: 